In [ ]:
#| default_exp model

## Model overview

This notebook implements a small PRAGMA-style encoder that consumes the tokenized batches from `fastpragma.dataloader`. The goal here is a clear working version before adding production details such as RoPE, optimized varlen attention, LoRA, or larger model configs.

Assumptions:

- Tokens are represented as `(key_id, val_id, val_pos)` triples.
- Profile and lifelong state are encoded together into one user representation.
- Events are packed by token and encoded independently before being gathered into user histories.
- Calendar features are projected with a small MLP and added to event representations.
- The history encoder contextualizes each user representation with that user’s event representations.
- The MLM head predicts masked event value tokens from token-level, event-level, and user-level context.
- The dataloader owns masking, labels, offsets, and padding masks; the model consumes that batch contract directly.

Each exported piece is followed by a small `test_eq` check so we can build the model incrementally and verify shapes as we go.

In [ ]:
#| export
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from fastcore.all import *
from fastpragma.dataloader import pragma_dl
from fastpragma.data import Tokenizer, PRAGMADataset

In [ ]:
test_eq(torch.tensor([1,2]).numel(), 2)

In [ ]:
#| export
class KVEmb(nn.Module):
    "Embed key, value, and value-position triples"
    def __init__(self, n_keys, n_vals, d_model, max_val_pos=32):
        super().__init__()
        store_attr()
        self.key_emb,self.val_emb,self.pos_emb = nn.Embedding(n_keys, d_model),nn.Embedding(n_vals, d_model),nn.Embedding(max_val_pos, d_model)

    def forward(self, x):
        k,v,p = x[...,0],x[...,1],x[...,2].clamp_max(self.max_val_pos-1)
        return self.key_emb(k) + self.val_emb(v) + self.pos_emb(p)

In [ ]:
emb = KVEmb(5, 11, 7)
x = torch.tensor([[[1,2,0], [3,4,1]]])
y = emb(x)
test_eq(y.shape, (1,2,7))

In [ ]:
#| export
def prepend_tok(x, tok):
    "Prepend one learned token to each batch item"
    return torch.cat([tok.expand(x.shape[0], 1, -1), x], 1)

In [ ]:
x = torch.randn(3, 4, 5)
tok = nn.Parameter(torch.randn(1, 1, 5))
y = prepend_tok(x, tok)
test_eq(y.shape, (3,5,5))
test_eq(y[:,0].shape, (3,5))

In [ ]:
#| export
class RotaryEmb(nn.Module):
    "Rotary positional embedding"
    def __init__(self, dim, base=10000):
        super().__init__()
        inv_freq = 1. / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)

    def forward(self, pos):
        freqs = pos.float()[...,None] * self.inv_freq
        return freqs.cos(),freqs.sin()

def rotate_half(x):
    x1,x2 = x[...,::2],x[...,1::2]
    return torch.stack([-x2,x1], -1).flatten(-2)

def apply_rope(x, cos, sin):
    cos,sin = cos.repeat_interleave(2, -1),sin.repeat_interleave(2, -1)
    return x*cos + rotate_half(x)*sin

#| export
def rope_apply(q, k, rope_emb, pos):
    cos,sin = rope_emb(pos)
    if cos.ndim == 2: cos,sin = cos[None,None],sin[None,None]
    else: cos,sin = cos[:,None],sin[:,None]
    return apply_rope(q, cos, sin),apply_rope(k, cos, sin)

In [ ]:
rope = RotaryEmb(8)
pos = torch.arange(5)
cos,sin = rope(pos)
x = torch.randn(2, 5, 8)
y = apply_rope(x, cos[None], sin[None])
test_eq(cos.shape, (5,4))
test_eq(sin.shape, (5,4))
test_eq(y.shape, x.shape)
test_close(y.norm(dim=-1), x.norm(dim=-1), eps=1e-5)

In [ ]:
#| export
class RopeMHA(nn.Module):
    "Multi-head attention with optional RoPE"
    def __init__(self, d_model, n_heads=4, p=0.1, rope=False):
        super().__init__()
        store_attr()
        self.qkv,self.proj,self.drop = nn.Linear(d_model, d_model*3),nn.Linear(d_model, d_model),nn.Dropout(p)
        self.rope_emb = RotaryEmb(d_model//n_heads) if rope else None

    def forward(self, x, mask=None, pos=None):
        bs,sl,_ = x.shape
        q,k,v = self.qkv(x).chunk(3, -1)
        q,k,v = [o.view(bs, sl, self.n_heads, -1).transpose(1, 2) for o in (q,k,v)]
        if self.rope_emb is not None:
            pos = ifnone(pos, torch.arange(sl, device=x.device))
            q,k = rope_apply(q, k, self.rope_emb, pos)
        attn_mask = None if mask is None else mask.bool()[:,None,None]
        y = nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.p if self.training else 0.)
        return self.proj(y.transpose(1, 2).contiguous().view(bs, sl, -1))

class EncoderLayer(nn.Module):
    "Pre-norm Transformer encoder layer"
    def __init__(self, d_model, n_heads=4, d_ff=None, p=0.1, rope=False):
        super().__init__()
        d_ff = ifnone(d_ff, d_model*4)
        self.attn,self.ln1,self.ln2,self.drop = RopeMHA(d_model, n_heads, p, rope),nn.LayerNorm(d_model),nn.LayerNorm(d_model),nn.Dropout(p)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(p), nn.Linear(d_ff, d_model))

    def forward(self, x, mask=None, pos=None):
        x = x + self.drop(self.attn(self.ln1(x), mask, pos))
        return x + self.drop(self.ff(self.ln2(x)))

class Encoder(nn.Module):
    "Small Transformer encoder wrapper"
    def __init__(self, d_model, n_heads=4, n_layers=2, d_ff=None, p=0.1, rope=False):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, p, rope) for _ in range(n_layers)])

    def forward(self, x, mask=None, pos=None):
        for l in self.layers: x = l(x, mask, pos)
        return x

In [ ]:
enc = Encoder(8, n_heads=2, n_layers=1)
x = torch.randn(3, 5, 8)
mask = torch.tensor([[1,1,1,1,1], [1,1,1,0,0], [1,1,0,0,0]]).bool()
y = enc(x, mask)
test_eq(y.shape, x.shape)

In [ ]:
#| export
class ProfileEncoder(nn.Module):
    "Encode profile and lifelong tokens into one user vector"
    def __init__(self, n_keys, n_vals, d_model, n_heads=4, n_layers=2):
        super().__init__()
        self.emb = KVEmb(n_keys, n_vals, d_model)
        self.usr = nn.Parameter(torch.randn(1, 1, d_model)*0.02)
        self.enc = Encoder(d_model, n_heads, n_layers, rope=True)

    def forward(self, profile, profile_mask, profile_time=None, lifelong=None, lifelong_mask=None, lifelong_time=None):
        x,mask = self.emb(profile),profile_mask.bool()
        t = torch.zeros_like(profile_mask, dtype=torch.float) if profile_time is None else profile_time.float()
        if lifelong is not None:
            x = torch.cat([x, self.emb(lifelong)], 1)
            mask = torch.cat([mask, lifelong_mask.bool()], 1)
            lt = torch.zeros_like(lifelong_mask, dtype=torch.float) if lifelong_time is None else lifelong_time.float()
            t = torch.cat([t, lt], 1)
        x = prepend_tok(x, self.usr)
        mask = torch.cat([torch.ones(mask.shape[0], 1, device=mask.device, dtype=torch.bool), mask], 1)
        t = torch.cat([torch.zeros(t.shape[0], 1, device=t.device), t], 1)
        return self.enc(x, mask, t)[:,0]

In [ ]:
prof = torch.randint(0, 5, (3, 4, 3))
life = torch.randint(0, 5, (3, 2, 3))
prof[...,1] %= 11
life[...,1] %= 11
prof_mask = torch.ones(3, 4).bool()
life_mask = torch.tensor([[1,1], [1,0], [0,0]]).bool()
prof_time = torch.zeros(3, 4)
life_time = torch.randn(3, 2).abs()
m = ProfileEncoder(5, 11, 8, n_heads=2, n_layers=1)
y = m(prof, prof_mask, prof_time, life, life_mask, life_time)
test_eq(y.shape, (3,8))

In [ ]:
#| export
def pad_packed(x, offsets):
    "Pad packed rows using offsets"
    xs = [x[s:e] for s,e in zip(offsets[:-1], offsets[1:])]
    return nn.utils.rnn.pad_sequence(xs, batch_first=True),torch.tensor([len(o) for o in xs], device=x.device)

In [ ]:
x = torch.randn(7, 5)
offsets = torch.tensor([0, 2, 5, 7])
y,lens = pad_packed(x, offsets)
test_eq(y.shape, (3,3,5))
test_eq(lens, torch.tensor([2,3,2]))

### Varlen attention

In [ ]:
#| export
try: from flash_attn.flash_attn_interface import flash_attn_varlen_qkvpacked_func
except Exception: flash_attn_varlen_qkvpacked_func = None

In [ ]:
#| export
def cu_seqlens(offsets): return offsets.to(torch.int32)
def has_flash_attn(): return flash_attn_varlen_qkvpacked_func is not None and torch.cuda.is_available()

In [ ]:
offs = torch.tensor([0, 2, 5, 7])
test_eq(cu_seqlens(offs), torch.tensor([0,2,5,7], dtype=torch.int32))
test_eq(has_flash_attn(), False)

In [ ]:
#| export
def lens_from_offsets(offsets): return offsets[1:] - offsets[:-1]

In [ ]:
offs = torch.tensor([0, 2, 5, 7])
test_eq(lens_from_offsets(offs), torch.tensor([2,3,2]))

In [ ]:
#| export
class VarlenMHA(nn.Module):
    "Packed multi-head attention with padded fallback"
    def __init__(self, d_model, n_heads=4, p=0.1):
        super().__init__()
        store_attr()
        self.qkv,self.proj = nn.Linear(d_model, d_model*3),nn.Linear(d_model, d_model)

    def forward(self, x, offsets):
        if has_flash_attn(): return self._flash(x, offsets)
        return self._padded(x, offsets)

    def _flash(self, x, offsets):
        qkv = self.qkv(x).view(x.shape[0], 3, self.n_heads, -1)
        y = flash_attn_varlen_qkvpacked_func(qkv, cu_seqlens(offsets), int(lens_from_offsets(offsets).max()), dropout_p=self.p if self.training else 0.)
        return self.proj(y.reshape(x.shape[0], -1))

    def _padded(self, x, offsets):
        xp,lens = pad_packed(x, offsets)
        bs,sl,_ = xp.shape
        q,k,v = self.qkv(xp).chunk(3, -1)
        q,k,v = [o.view(bs, sl, self.n_heads, -1).transpose(1, 2) for o in (q,k,v)]
        mask = torch.arange(sl, device=x.device)[None] < lens[:,None]
        y = nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=mask[:,None,None], dropout_p=self.p if self.training else 0.)
        y = self.proj(y.transpose(1, 2).contiguous().view(bs, sl, -1))
        return torch.cat([y[i,:l] for i,l in enumerate(lens)])

In [ ]:
m = VarlenMHA(8, n_heads=2)
x = torch.randn(7, 8)
offs = torch.tensor([0, 2, 5, 7])
y = m(x, offs)
test_eq(y.shape, x.shape)
test_eq(has_flash_attn(), False)

In [ ]:
#| export
class VarlenEncoderLayer(nn.Module):
    "Pre-norm packed Transformer encoder layer"
    def __init__(self, d_model, n_heads=4, d_ff=None, p=0.1):
        super().__init__()
        d_ff = ifnone(d_ff, d_model*4)
        self.attn,self.ln1,self.ln2,self.drop = VarlenMHA(d_model, n_heads, p),nn.LayerNorm(d_model),nn.LayerNorm(d_model),nn.Dropout(p)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(p), nn.Linear(d_ff, d_model))

    def forward(self, x, offsets):
        x = x + self.drop(self.attn(self.ln1(x), offsets))
        return x + self.drop(self.ff(self.ln2(x)))

class VarlenEncoder(nn.Module):
    "Packed Transformer encoder wrapper"
    def __init__(self, d_model, n_heads=4, n_layers=2, d_ff=None, p=0.1):
        super().__init__()
        self.layers = nn.ModuleList([VarlenEncoderLayer(d_model, n_heads, d_ff, p) for _ in range(n_layers)])

    def forward(self, x, offsets):
        for l in self.layers: x = l(x, offsets)
        return x

In [ ]:
m = VarlenEncoder(8, n_heads=2, n_layers=1)
x = torch.randn(7, 8)
offs = torch.tensor([0, 2, 5, 7])
y = m(x, offs)
test_eq(y.shape, x.shape)

In [ ]:
#| export
def prepend_packed_tok(x, offsets, tok):
    "Prepend one learned token to each packed sequence"
    xs = [torch.cat([tok[0], x[s:e]]) for s,e in zip(offsets[:-1], offsets[1:])]
    return torch.cat(xs),torch.cat([offsets.new_tensor([0]), torch.tensor([len(o) for o in xs], device=offsets.device).cumsum(0)])

In [ ]:
x = torch.randn(7, 8)
tok = nn.Parameter(torch.randn(1, 1, 8))
offs = torch.tensor([0, 2, 5, 7])
y,new_offs = prepend_packed_tok(x, offs, tok)
test_eq(y.shape, (10,8))
test_eq(new_offs, torch.tensor([0,3,7,10]))

In [ ]:
#| export
class EventEncoder(nn.Module):
    "Encode packed event tokens into event and token representations"
    def __init__(self, n_keys, n_vals, d_model, n_heads=4, n_layers=2):
        super().__init__()
        self.emb = KVEmb(n_keys, n_vals, d_model)
        self.evt = nn.Parameter(torch.randn(1, 1, d_model)*0.02)
        self.enc = VarlenEncoder(d_model, n_heads, n_layers)

    def forward(self, event_tokens, event_offsets):
        x,offs = prepend_packed_tok(self.emb(event_tokens), event_offsets, self.evt)
        h = self.enc(x, offs)
        lens = lens_from_offsets(offs)
        evt = h[offs[:-1]]
        tok = torch.cat([h[s+1:e] for s,e in zip(offs[:-1], offs[1:])])
        return evt,tok

In [ ]:
ev = torch.tensor([[1,2,0], [2,3,0], [3,4,0], [1,5,0], [2,6,0], [1,7,0], [2,8,0]])
offs = torch.tensor([0, 2, 5, 7])
m = EventEncoder(5, 11, 8, n_heads=2, n_layers=1)
evt,tok = m(ev, offs)
test_eq(evt.shape, (3,8))
test_eq(tok.shape, (7,8))

In [ ]:
#| export
class CalEmb(nn.Module):
    "Embed event calendar features"
    def __init__(self, n_cal, d_model):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(n_cal, d_model), nn.GELU(), nn.Linear(d_model, d_model))

    def forward(self, x): return self.mlp(x.float())

In [ ]:
m = CalEmb(3, 8)
x = torch.randn(5, 3)
y = m(x)
test_eq(y.shape, (5,8))

In [ ]:
#| export
def pad_history(z_usr, z_evt, event_time, history_offsets):
    "Build padded user histories and history times"
    xs = [torch.cat([z_usr[i][None], z_evt[s:e]]) for i,(s,e) in enumerate(zip(history_offsets[:-1], history_offsets[1:]))]
    ts = [torch.cat([event_time.new_zeros(1), event_time[s:e].float()]) for s,e in zip(history_offsets[:-1], history_offsets[1:])]
    return pad_sequence(xs, batch_first=True),torch.tensor([len(o) for o in xs], device=z_evt.device),pad_sequence(ts, batch_first=True)

In [ ]:
z_usr,z_evt = torch.randn(3, 8),torch.randn(7, 8)
event_time = torch.arange(7).float()
offs = torch.tensor([0, 2, 5, 7])
x,lens,pos = pad_history(z_usr, z_evt, event_time, offs)
test_eq(x.shape, (3,4,8))
test_eq(lens, torch.tensor([3,4,3]))
test_eq(pos.shape, (3,4))
test_eq(pos[1], torch.tensor([0.,2.,3.,4.]))


In [ ]:
#| export
class HistoryEncoder(nn.Module):
    "Contextualize user histories"
    def __init__(self, d_model, n_heads=4, n_layers=2):
        super().__init__()
        self.enc = Encoder(d_model, n_heads, n_layers, rope=True)

    def forward(self, z_usr, z_evt, event_user, history_offsets, event_time):
        x,lens,pos = pad_history(z_usr, z_evt, event_time, history_offsets)
        mask = torch.arange(x.shape[1], device=x.device)[None] < lens[:,None]
        h = self.enc(x, mask, pos)
        return h[:,0],torch.cat([h[i,1:l] for i,l in enumerate(lens)])

In [ ]:
m = HistoryEncoder(8, n_heads=2, n_layers=1)
z_usr,z_evt = torch.randn(3, 8),torch.randn(7, 8)
event_user = torch.tensor([0,0,1,1,1,2,2])
event_time = torch.arange(7).float()
offs = torch.tensor([0, 2, 5, 7])
h_usr,h_evt = m(z_usr, z_evt, event_user, offs, event_time)
test_eq(h_usr.shape, (3,8))
test_eq(h_evt.shape, (7,8))

In [ ]:
#| export
class MLMHead(nn.Module):
    "Predict masked event value tokens"
    def __init__(self, d_model, n_vals):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model*3, d_model), nn.GELU(), nn.Linear(d_model, n_vals))

    def forward(self, tok_h, evt_h, usr_h): return self.net(torch.cat([tok_h, evt_h, usr_h], -1))

In [ ]:
m = MLMHead(8, 11)
tok_h,evt_h,usr_h = torch.randn(5,8),torch.randn(5,8),torch.randn(5,8)
y = m(tok_h, evt_h, usr_h)
test_eq(y.shape, (5,11))

In [ ]:
#| export
def token_event_ids(event_offsets):
    "Map packed event tokens to event ids"
    lens = event_offsets[1:] - event_offsets[:-1]
    return torch.repeat_interleave(torch.arange(len(lens), device=event_offsets.device), lens)

In [ ]:
offs = torch.tensor([0, 2, 5, 7])
ids = token_event_ids(offs)
test_eq(ids, torch.tensor([0,0,1,1,1,2,2]))

In [ ]:
#| export
def token_user_ids(event_offsets, event_user):
    "Map packed event tokens to user ids"
    return event_user[token_event_ids(event_offsets)]

In [ ]:
offs = torch.tensor([0, 2, 5, 7])
event_user = torch.tensor([0,0,2])
ids = token_user_ids(offs, event_user)
test_eq(ids, torch.tensor([0,0,0,0,0,2,2]))

Varlen attention and RoPE are used in different parts of the model. The Event Encoder uses packed/varlen attention to let tokens attend only within each event, without temporal RoPE. Profile and History encoders use RoPE because they have meaningful temporal coordinates: lifelong/profile timing for profile state, and time-to-last-event for event histories. Calendar features are projected separately and added to event representations before history encoding.

In [ ]:
#| export
class PRAGMAModel(nn.Module):
    "Small PRAGMA-style encoder"
    def __init__(self, n_keys, n_vals, d_model=128, n_heads=4, n_layers=2, n_cal=3):
        super().__init__()
        store_attr()
        self.profile_enc = ProfileEncoder(n_keys, n_vals, d_model, n_heads, n_layers)
        self.event_enc = EventEncoder(n_keys, n_vals, d_model, n_heads, n_layers)
        self.cal_emb = CalEmb(n_cal, d_model)
        self.history_enc = HistoryEncoder(d_model, n_heads, n_layers)
        self.mlm = MLMHead(d_model, n_vals)

    def forward(self, b):
        z_usr = self.profile_enc(b['profile'], b['profile_mask'], b.get('profile_time'), b.get('lifelong'), b.get('lifelong_mask'), b.get('lifelong_time'))
        z_evt,tok_h = self.event_enc(b['event_tokens'], b['event_offsets'])
        
        z_evt = z_evt + self.cal_emb(b['cal'])
        h_usr,h_evt = self.history_enc(z_usr, z_evt, b['event_user'], b['history_offsets'], b['event_time'])
        tok_evt,tok_usr = token_event_ids(b['event_offsets']),token_user_ids(b['event_offsets'], b['event_user'])
        m = b['mlm_mask'].bool()
        logits = self.mlm(tok_h[m], h_evt[tok_evt[m]], h_usr[tok_usr[m]])
        return dict(h_usr=h_usr, h_evt=h_evt, logits=logits, labels=b['event_labels'][m])

In [ ]:
bs,n_events,n_toks,d = 3,7,12,8
b = dict(
    profile=torch.randint(0, 5, (bs, 4, 3)), profile_mask=torch.ones(bs, 4).bool(), profile_time=torch.zeros(bs, 4),
    lifelong=torch.randint(0, 5, (bs, 2, 3)), lifelong_mask=torch.ones(bs, 2).bool(), lifelong_time=torch.randn(bs, 2).abs(),
    event_tokens=torch.randint(0, 5, (n_toks, 3)), event_offsets=torch.tensor([0,2,5,7,8,10,11,12]),
    event_user=torch.tensor([0,0,1,1,1,2,2]), history_offsets=torch.tensor([0,2,5,7]), event_time=torch.arange(n_events).float(),
    cal=torch.randn(n_events, 3), mlm_mask=torch.tensor([1,0,1,0,0,1,0,1,0,0,1,0]).bool(), event_labels=torch.randint(0, 11, (n_toks,)))
b['profile'][...,1] %= 11
b['lifelong'][...,1] %= 11
b['event_tokens'][...,1] %= 11
m = PRAGMAModel(5, 11, d_model=d, n_heads=2, n_layers=1)
out = m(b)
test_eq(out['h_usr'].shape, (bs,d))
test_eq(out['h_evt'].shape, (n_events,d))
test_eq(out['logits'].shape, (b['mlm_mask'].sum(),11))
test_eq(out['labels'].shape, (b['mlm_mask'].sum(),))

In [ ]:
out['h_usr'].shape == (bs,d)
out['h_evt'].shape == (n_events,d)
out['logits'].shape == (b['mlm_mask'].sum(),11)
out['labels'].shape == (b['mlm_mask'].sum(),)

True

In [ ]:
out = Path('data/ml100k_pragma')
shards = sorted(out.glob('shard_*.parquet'))
len(shards),shards[0]

(5, Path('data/ml100k_pragma/shard_0.parquet'))

In [ ]:
tok = Tokenizer.load(out/'tokenizer.json')
dl = pragma_dl(shards, max_tokens=12000, shuffle=False, mask=True, tok=tok)
b = next(iter(dl))
test_eq(set('profile profile_mask profile_time lifelong lifelong_mask lifelong_time event_tokens event_offsets event_user event_time cal history_offsets uids event_labels mlm_mask'.split()) <= set(b), True)

In [ ]:
n_keys,n_vals = len(tok.key_vocab),len(tok.val_vocab)
n_keys,n_vals

(11, 2604)

In [ ]:
#| export
def mlm_loss(out): return nn.functional.cross_entropy(out['logits'], out['labels'])

In [ ]:
tiny_dl = pragma_dl(shards, max_tokens=1500, shuffle=False, mask=True, tok=tok)
b = next(iter(tiny_dl))
m = PRAGMAModel(n_keys, n_vals, d_model=32, n_heads=2, n_layers=1)
outp = m(b)
loss = mlm_loss(outp)
loss.backward()
outp['h_usr'].shape,outp['h_evt'].shape,outp['logits'].shape,loss

(torch.Size([7, 32]),
 torch.Size([712, 32]),
 torch.Size([365, 2604]),
 tensor(8.1631, grad_fn=<NllLossBackward0>))

### using fastai

In [ ]:
#| export
import fastpragma.dataloader as fdl
from fastai.callback.core import Callback, CancelEpochException
from fastai.data.core import DataLoaders
from fastai.learner import Learner
from fastai.optimizer import Adam
from fastpragma.dataloader import PRAGMADataLoader

In [ ]:
#| export
class FastaiPRAGMADL:
    "Wrap PRAGMA batches as fastai input-target pairs"
    def __init__(self, dl): self.dl = dl
    def __iter__(self):
        for b in self.dl: yield b,b
    def __len__(self): return len(self.dl)

def fastai_pragma_dl(*args, **kwargs): return FastaiPRAGMADL(pragma_dl(*args, **kwargs))

In [ ]:
#| export
class PRAGMAWrapper(nn.Module):
    "Return MLM logits"
    def __init__(self, model): super().__init__(); self.model = model
    def forward(self, b): return self.model(b)['logits']

def pragma_loss(logits, b): return nn.functional.cross_entropy(logits, b['event_labels'][b['mlm_mask'].bool()])

In [ ]:
#| export
def pragma_dls(shards, tok, max_tokens=1500):
    train_dl = fastai_pragma_dl(shards, max_tokens=max_tokens, shuffle=True, mask=True, tok=tok)
    valid_dl = fastai_pragma_dl(shards[:1], max_tokens=max_tokens, shuffle=False, mask=True, tok=tok)
    return DataLoaders(train_dl, valid_dl)

def pragma_learner(dls, n_keys, n_vals, d_model=32, n_heads=2, n_layers=1, opt_func=Adam):
    m = PRAGMAModel(n_keys, n_vals, d_model=d_model, n_heads=n_heads, n_layers=n_layers)
    return Learner(dls, PRAGMAWrapper(m), loss_func=pragma_loss, opt_func=opt_func)

In [ ]:
dls = pragma_dls(shards, tok)
learn = pragma_learner(dls, n_keys, n_vals)
xb,yb = next(iter(dls.train))
loss = learn.loss_func(learn.model(xb), yb)
test_eq(loss.ndim, 0)
loss

tensor(7.8107, grad_fn=<NllLossBackward0>)

In [ ]:
class ShortEpochCB(Callback):
    def __init__(self, n=3): self.n = n
    def after_batch(self):
        if self.iter+1 >= self.n: raise CancelEpochException

In [ ]:
learn = pragma_learner(dls, n_keys, n_vals)
learn.fit(1, lr=1e-3, cbs=ShortEpochCB(3))

[0, '00:44']


In [ ]:
import nbdev; nbdev.nbdev_export()